# 01 · Data exploration

Describes the data for the report's Data section: the NRK-Quiz-QA questions and the Bokmål Wikipedia corpus with its chunks. Only word counts are read from the large Wikipedia files, so the notebook runs in under a minute.

**Reads:** `data/processed/questions.parquet`, `articles.parquet`, `chunks.parquet`
**Produces:** the tables `results/tables/data_*.csv` and the figure `results/figures/article_length.png`

Sections:
1. Questions: quizzes, options per question (and so the chance level), which letter is correct, question length
2. Wikipedia: articles, article length (the reason for 100-word chunks), chunks
3. Questions that mention recent years (the reason for the March 2025 dump)

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from norrag.config import get_path, load_config
from norrag.metrics import chance_accuracy

config = load_config()
processed = get_path(config, "data_processed")
tables, figures = get_path(config, "tables"), get_path(config, "figures")

questions = pd.read_parquet(processed / "questions.parquet")
articles = pd.read_parquet(processed / "articles.parquet", columns=["article_id", "n_words"])
chunks = pd.read_parquet(processed / "chunks.parquet", columns=["article_id", "n_words"])
print(f"{len(questions):,} questions, {len(articles):,} articles, {len(chunks):,} chunks")

## 1 · Questions

In [ ]:
per_quiz = questions.groupby("quiz").size()
question_words = questions.question.str.split().str.len()
overview = pd.Series({
    "questions": len(questions),
    "quizzes": questions.quiz.nunique(),
    "questions per quiz, median": per_quiz.median(),
    "questions per quiz, min-max": f"{per_quiz.min()}-{per_quiz.max()}",
    "words per question, median": question_words.median(),
    "words per question, mean": round(question_words.mean(), 1),
    "words per question, min-max": f"{question_words.min()}-{question_words.max()}",
})
overview.to_frame("value").to_csv(tables / "data_questions.csv")
overview.to_frame("value")

In [ ]:
# Options per question: a question with n options is guessed right 1/n of the time
n_options = questions.options.str.len()
options_table = n_options.value_counts().sort_index().rename_axis("options").to_frame("questions")
options_table["share"] = (options_table.questions / len(questions)).round(3)
options_table["chance"] = (1 / options_table.index).round(3)
options_table.to_csv(tables / "data_options.csv")
print(f"Chance level over all questions: {chance_accuracy(n_options):.1%}")
options_table

In [ ]:
# Which letter is the correct answer, by the number of options; with no pattern, each letter is right 1/n of the time
letters = pd.crosstab(n_options.rename("options"), questions.answer.rename("correct letter"))
letter_shares = letters.div(letters.sum(axis=1), axis=0).round(3)
letters.to_csv(tables / "data_answer_letters.csv")
letter_shares

## 2 · Wikipedia

Article length decides how well 100-word chunks fit: an article shorter than a chunk becomes one chunk, a long one is cut into overlapping pieces (20 words of overlap).

In [ ]:
chunks_per_article = chunks.groupby("article_id").size()
wikipedia = pd.Series({
    "articles": len(articles),
    "words in total": int(articles.n_words.sum()),
    "words per article, median": articles.n_words.median(),
    "articles under 100 words": f"{(articles.n_words < 100).mean():.1%}",
    "articles under 200 words": f"{(articles.n_words < 200).mean():.1%}",
    "chunks": len(chunks),
    "words per chunk, median": chunks.n_words.median(),
    "words per chunk, mean": round(chunks.n_words.mean(), 1),
    "chunks per article, median": chunks_per_article.median(),
    "chunks per article, max": chunks_per_article.max(),
})
wikipedia.to_frame("value").to_csv(tables / "data_wikipedia.csv")
wikipedia.to_frame("value")

In [ ]:
# Figure: article length on a log scale, with the 100-word chunk size marked
INK, INK_2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.edgecolor": AXIS,
                     "xtick.color": INK_2, "ytick.color": INK_2, "savefig.dpi": 300, "savefig.bbox": "tight"})
lengths = articles.n_words[articles.n_words > 0]
bins = np.logspace(0, np.log10(lengths.max()), 60)  # equal widths on a log scale
fig, ax = plt.subplots(figsize=(6.5, 3.0))
counts, _, _ = ax.hist(lengths, bins=bins, color=MUTED, edgecolor="white", linewidth=0.5, zorder=2)
ax.set_xscale("log")
ax.set_xlim(5, lengths.max() * 1.2)  # the 200 articles under 5 words are too few to see as bars anyway
ax.set_ylim(0, counts.max() * 1.2)  # room for the label above the bars
ax.axvline(100, color=INK, linewidth=1, zorder=3)
ax.text(100 * 1.12, counts.max() * 1.17, f"100 words (chunk size): {(lengths < 100).mean():.0%} of articles are shorter",
        va="top", color=INK, fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda value, position: f"{value:,.0f}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, position: f"{value:,.0f}"))
ax.set_xlabel("Words per article (log scale)")
ax.set_ylabel("Articles")
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(figures / "article_length.png")
plt.show()

## 3 · Questions that mention recent years

A question about events after the Wikipedia dump cannot be answered from it. A year written in the question or its options is a rough sign of how recent the question is: it can also be a year in the past that the question asks about. The thresholds are the dates of the dumps considered: March 2019 (the first one), November 2023 (Hugging Face) and March 2025 (the one used).

In [ ]:
def latest_year(row):
    # The latest year from 1900 to 2099 written in the question or its options (None if there is none)
    text = " ".join([row.question, *row.options])
    years = [int(year) for year in re.findall(r"\b(?:19|20)\d{2}\b", text)]
    return max(years) if years else None

year = questions.apply(latest_year, axis=1)
recent = pd.Series({
    "mention a year": int(year.notna().sum()),
    "mention 2019 or later (after the March 2019 dump)": int((year >= 2019).sum()),
    "mention 2024 or later (after the November 2023 dump)": int((year >= 2024).sum()),
    "mention 2025 or later (after the March 2025 dump)": int((year >= 2025).sum()),
}).to_frame("questions")
recent["share"] = (recent.questions / len(questions)).round(3)
recent.to_csv(tables / "data_years.csv")
recent